In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
PROJECT_ROOT = Path("D:/Dev/enterprise-aws-data-lakehouse-ml-system")

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

In [3]:
train_transaction = pd.read_parquet(DATA_RAW / "train_transaction.parquet")
train_identity = pd.read_parquet(DATA_RAW / "train_identity.parquet")

print(train_transaction.shape)
print(train_identity.shape)

(590540, 394)
(144233, 41)


In [4]:
train_transaction 

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,V330,V331,V332,V333,V334,V335,V336,V337,V338,V339
0,2987000,0,86400,68.50,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.00,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.00,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.00,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.00,H,4497,514.0,150.0,mastercard,102.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,W,6550,NaN,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590536,3577536,0,15811049,39.50,W,10444,225.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590537,3577537,0,15811079,30.95,W,12037,595.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590538,3577538,0,15811088,117.00,W,7826,481.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
train = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print(train.shape)


(590540, 434)


### Verify Target + Core Columns

In [6]:
assert "isFraud" in train.columns
assert "TransactionDT" in train.columns

### Build Time Foundation

In [7]:
SECONDS_IN_DAY = 24 * 60 * 60            # TransactionDT: Number of seconds from a reference start date

train["day"] = train["TransactionDT"] // SECONDS_IN_DAY 
train["hour"] = (train["TransactionDT"] // 3600) % 24  # Extract transaction hour (0–23) from TransactionDT seconds

In [8]:
train["day"].describe()


count    590540.000000
mean         84.729199
std          53.437277
min           1.000000
25%          35.000000
50%          84.000000
75%         130.000000
max         182.000000
Name: day, dtype: float64

In [9]:
train["hour"].value_counts().sort_index()

hour
0     37795
1     32797
2     26732
3     20802
4     14839
5      9701
6      6007
7      3704
8      2591
9      2479
10     3627
11     6827
12    12451
13    20315
14    28328
15    33859
16    38698
17    40723
18    41639
19    42115
20    41782
21    41641
22    41139
23    39949
Name: count, dtype: int64

In [10]:
train

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo,day,hour
0,2987000,0,86400,68.50,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
1,2987001,0,86401,29.00,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
2,2987002,0,86469,59.00,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
3,2987003,0,86499,50.00,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,0
4,2987004,0,86506,50.00,H,4497,514.0,150.0,mastercard,102.0,...,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,W,6550,NaN,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,182,23
590536,3577536,0,15811049,39.50,W,10444,225.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,182,23
590537,3577537,0,15811079,30.95,W,12037,595.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,182,23
590538,3577538,0,15811088,117.00,W,7826,481.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,182,23


### Basic Frequency Encoding

In [11]:
# Loop through selected card-related categorical columns
for col in ["card1", "card2", "card3", "card4"]:
    freq = train[col].value_counts(dropna=False)    # Calculate frequency of each unique value in the column
    train[f"{col}_freq"] = train[col].map(freq)     # dropna=False ensures missing values (NaN) are also counted
                                                    # Map each value to its frequency; unseen values become NaN (→ NaN)

    # Map each original value to its corresponding frequency
    # This creates a new feature: <column_name>_freq  (Create new features. any number is converted to non zero value)
    # Example: card1 -> card1_freq  


In [12]:
train

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_37,id_38,DeviceType,DeviceInfo,day,hour,card1_freq,card2_freq,card3_freq,card4_freq
0,2987000,0,86400,68.50,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,1,0,43,8933,521287,6651
1,2987001,0,86401,29.00,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,1,0,683,3056,521287,189217
2,2987002,0,86469,59.00,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,1,0,1108,38145,521287,384767
3,2987003,0,86499,50.00,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,1,0,4209,6137,521287,189217
4,2987004,0,86506,50.00,H,4497,514.0,150.0,mastercard,102.0,...,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M,1,0,18,14541,521287,189217
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,3577535,0,15811047,49.00,W,6550,NaN,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,182,23,1183,8933,521287,384767
590536,3577536,0,15811049,39.50,W,10444,225.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,182,23,12,7445,521287,189217
590537,3577537,0,15811079,30.95,W,12037,595.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,182,23,690,734,521287,189217
590538,3577538,0,15811088,117.00,W,7826,481.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,182,23,3006,6336,521287,189217


In [17]:
# Filter rows where card1 = 13926 and card1_freq = 43
result_df = train[
    (train["card1"] == 13926) & 
    (train["card1_freq"] == 43)
][[ "card1", "card1_freq"]]

result_df

,card1,card1_freq
0,13926,43
36492,13926,43
39562,13926,43
46460,13926,43
55544,13926,43
92759,13926,43
104924,13926,43
136584,13926,43
136766,13926,43
159934,13926,43


In [13]:
train.shape

(590540, 440)

In [14]:
train.to_parquet(
    DATA_PROCESSED / "train_foundation.parquet",
    index=False
)

## Identity & Time Foundation – Key Findings

### Dataset Overview

- **Train Transaction Shape:** 590,540 rows × 394 columns  
- **Train Identity Shape:** 144,233 rows × 41 columns  
- **Merged Dataset Shape:** 590,540 rows × 434 columns  
- Merge performed using `TransactionID` with `left join`.

✅ No row loss during merge.  
✅ Target column `isFraud` verified.  
✅ Core column `TransactionDT` verified.

---

### Temporal Foundation

Derived new time-based behavioral features:

- `day = TransactionDT // 86400`
- `hour = (TransactionDT // 3600) % 24`

### Observations:

- Dataset spans multiple distinct days (time progression confirmed).
- Hourly distribution created to enable:
  - Night fraud pattern detection
  - Behavioral time modeling
  - Temporal stability validation

This converts raw seconds into interpretable behavioral time units.

---

### Frequency Encoding (Identity Signal Strengthening)

Applied frequency encoding to:

- `card1`
- `card2`
- `card3`
- `card4`

New features created:

- `card1_freq`
- `card2_freq`
- `card3_freq`
- `card4_freq`

### Purpose:

- Capture transaction density per identity
- Normalize high-cardinality categorical variables
- Improve generalization across train/test distribution shifts

This aligns with Kaggle winning strategies for fraud modeling.

---

## Next Stage

Proceed to:

`03_behavioral_aggregation_engine.ipynb`

Focus:

- UID construction
- User-level aggregation features
- Time-gap modeling
- Behavioral deviation metrics

Fraud detection modeling begins from this layer.